# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Adtividad en Equipos: sistema LLM + RAG**

* **Nombres y matrículas:**

  *   Jair Alejandro del Rosario German A01734802
  *   Alejandro Ramos Contreras A01747461
  *   Mariana Favarony Avila A01704671
  *   Manuel Alejandro Cardona Esquivel A01795954

* ##### **El formato de este cuaderno de Jupyter es libre, pero debe incuir al menos las siguientes secciones:**

  * ##### **Introducción de la problemática a resolver.**
  * ##### **Sistema RAG + LLM**
  * ##### **El chatbot, incluyendo ejemplos de prueba.**
  * ##### **Conclusiones**

* ##### **Pueden importar los paquetes o librerías que requieran.**

* ##### **Pueden incluir las celdas y líneas de código que deseen.**

## Chatbot RAG para consulta de informes trimestrales de Banco de México

## 1. Definición del Problema

Los informes trimestrales publicados por Banco de México constituyen una fuente  de información para comprender la evolución de la economía mexicana, las perspectivas de inflación, los riesgos macroeconómicos y las condiciones financieras del país. Sin embargo, estos documentos suelen tener una extensión considerable y contienen información técnica distribuida a lo largo de múltiples capítulos, secciones y anexos.

Como consecuencia, localizar información específica dentro de varios reportes puede convertirse en una tarea compleja y poco eficiente para analistas, investigadores, estudiantes y tomadores de decisiones. La consulta tradicional requiere revisar manualmente grandes volúmenes de información, lo que incrementa el tiempo necesario para encontrar datos relevantes y dificulta el aprovechamiento del conocimiento contenido en los documentos.

Para atender esta problemática se propone el desarrollo de un chatbot basado en la arquitectura Retrieval-Augmented Generation (RAG), capaz de responder preguntas en lenguaje natural utilizando como fuente de conocimiento los Informes Trimestrales 2025 de Banco de México.

---

## 2. Justificación del uso de RAG

Los Modelos de Lenguaje de Gran Escala (LLM) poseen una alta capacidad para generar texto y responder preguntas. Sin embargo, cuando se utilizan de forma aislada presentan limitaciones importantes, ya que sus respuestas dependen exclusivamente de la información disponible durante su entrenamiento y pueden generar información incorrecta o no fundamentada.

La arquitectura Retrieval-Augmented Generation (RAG) resuelve esta limitación incorporando un mecanismo de recuperación de información documental antes de generar la respuesta. De esta manera, el modelo consulta primero los documentos relevantes y posteriormente utiliza dicha información para construir una respuesta contextualizada.

La incorporación de RAG resulta especialmente adecuada para este proyecto debido a que los informes trimestrales contienen información especializada que cambia con el tiempo y que debe ser consultada directamente desde la fuente documental para garantizar precisión y trazabilidad.

---

## 3. Selección del modelo de lenguaje

Para la generación de respuestas se seleccionó Llama 3.2, ejecutado localmente mediante Ollama.

La elección de este modelo responde a tres factores principales:

* Es un modelo de código abierto que puede ejecutarse localmente sin depender de servicios externos.
* Presenta un balance adecuado entre capacidad de razonamiento, velocidad de respuesta y requerimientos computacionales.
* Permite integrarse de forma eficiente dentro de una arquitectura RAG donde la precisión depende principalmente de la calidad de la recuperación documental.

Dado que el objetivo principal del sistema consiste en responder preguntas fundamentadas en los documentos de Banco de México, Llama 3.2 proporciona capacidades suficientes para sintetizar la información recuperada y generar respuestas claras, coherentes y contextualizadas.

---

## 4. Objetivo del proyecto

Desarrollar un chatbot basado en Recuperación Aumentada por Generación (RAG) que permita consultar los Informes Trimestrales 2025 de Banco de México mediante preguntas formuladas en lenguaje natural.

La solución integra documentos PDF, embeddings semánticos, una base vectorial ChromaDB y un Modelo de Lenguaje de Gran Escala (LLM), permitiendo recuperar información relevante y generar respuestas fundamentadas en evidencia documental.

---

## 5. Arquitectura de la solución

La solución implementada sigue una arquitectura Retrieval-Augmented Generation compuesta por dos grandes etapas: construcción de la base de conocimiento y generación de respuestas.

Flujo general del sistema:

Informes PDF → Conversión a Texto → Limpieza → Segmentación (Chunks) → Embeddings → ChromaDB → Recuperación Semántica → LLM → Respuesta

Durante la primera etapa los documentos son transformados en representaciones vectoriales que permiten realizar búsquedas semánticas eficientes. Posteriormente, cuando un usuario realiza una consulta, el sistema identifica los fragmentos más relevantes y los proporciona como contexto al modelo de lenguaje para generar una respuesta fundamentada.

---

## 6. Metodología de implementación

El desarrollo del sistema se divide en dos fases principales.

### Fase 1: Construcción de la base de conocimiento

Durante esta fase se procesan los Informes Trimestrales de Banco de México para convertirlos en una base vectorial consultable.

Las actividades principales son:

1. Conversión de documentos PDF a texto estructurado.
2. Limpieza y normalización del contenido.
3. División de documentos en fragmentos de información (chunks).
4. Generación de embeddings semánticos.
5. Almacenamiento de embeddings en ChromaDB.

### Fase 2: Implementación del chatbot

Una vez construida la base de conocimiento, se implementa el flujo de consulta del chatbot:

1. El usuario formula una pregunta.
2. La pregunta es transformada en un embedding.
3. ChromaDB recupera los fragmentos más relevantes.
4. Los fragmentos recuperados son incorporados al prompt.
5. Llama 3.2 genera una respuesta basada exclusivamente en la información recuperada.

Esta estrategia permite reducir alucinaciones, mejorar la precisión de las respuestas y mantener trazabilidad respecto a la información utilizada para construir cada respuesta.


In [6]:
!pip3 install --quiet langchain langchain-community chromadb sentence-transformers pypdf langchain-huggingface langchain-ollama langchain-text-splitters

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [7]:
from pypdf import PdfReader

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import os

/Users/alejandrodelrosario/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [10]:
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
import re

In [11]:
# Configuración de rutas locales (ejecución en VS Code / entorno local con Ollama)
BASE_DIR = Path("/Users/alejandrodelrosario/Downloads/resources")

pdf_folder = str(BASE_DIR / "data")

print(os.listdir(pdf_folder))

['reporteOctubreDiciembre2025.pdf', 'reporteJulioSeptiembre2025.pdf', 'reporteEneroMarzo2025.pdf', 'reporteAbrilJunio2025.pdf']


## Definición de Fuentes de Información

Los documentos utilizados en este proyecto corresponden a los informes trimestrales del año del 2025 de Banco de México.

Durante esta etapa se definen las rutas de almacenamiento y los documentos que serán procesados para la construcción de la base vectorial.


In [12]:
DATA_DIR = BASE_DIR / "data"

OUT_DIR = BASE_DIR / "markdown"

CHROMA_DIR = BASE_DIR / "chroma_db"

PDFS = [
    ("2025-Q1", "reporteEneroMarzo2025.pdf"),
    ("2025-Q2", "reporteAbrilJunio2025.pdf"),
    ("2025-Q3", "reporteJulioSeptiembre2025.pdf"),
    ("2025-Q4", "reporteOctubreDiciembre2025.pdf"),
]

# Conversión de Documentos PDF a Texto Procesable
Los reportes originales se encuentran en formato PDF. El primer paso consiste en convertir los documentos a un formato textual estructurado que permita posteriormente realizar tareas de limpieza, segmentación y generación de embeddings.

El resultado de esta etapa será una colección de archivos en formato Markdown que preservan el contenido de los reportes y facilitan su procesamiento posterior.


In [13]:
def pdfs_to_markdown():

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    markdown_files = []

    for quarter, filename in PDFS:

        pdf_path = DATA_DIR / filename

        if not pdf_path.exists():
            print(f"[AVISO] No encontrado: {pdf_path}, se omite.")
            continue

        md_path = OUT_DIR / f"{quarter}.md"

        print(f"Convirtiendo {filename}")

        reader = PdfReader(str(pdf_path))

        text_parts = [page.extract_text() or "" for page in reader.pages]

        md_path.write_text(
            "\n".join(text_parts),
            encoding="utf-8"
        )

        markdown_files.append((quarter, md_path))

    return markdown_files

# Limpieza y normalización de la información

Una vez convertido el contenido a formato textual, se realiza un proceso de limpieza y normalización con el objetivo de eliminar elementos que no aportan valor al análisis semántico.

Durante esta etapa se eliminan separadores de tablas, caracteres redundantes y problemas de formato derivados de la conversión desde PDF.


In [14]:
def clean_markdown(text):

    lines = text.splitlines()

    cleaned = []

    for line in lines:

        stripped = line.strip()

        if re.fullmatch(r"[\|\-\s:]+", stripped):
            continue

        if stripped.count("|") >= 6:
            stripped = stripped.replace("|", " ")

        stripped = re.sub(
            r"\s+",
            " ",
            stripped
        ).strip()

        if stripped:
            cleaned.append(stripped)

    return "\n".join(cleaned)

# Segmentación de los documentos (Chunking)

Los informes trimestrales contienen una gran cantidad de información distribuida a lo largo de cientos de páginas. Debido a las limitaciones de contexto presentes en los modelos de lenguaje y los sistemas de recuperación, resulta poco eficiente trabajar con documentos completos.

Por esta razón, los documentos son divididos en fragmentos más pequeños denominados chunks.

Adicionalmente, se incorpora un mecanismo de traslape (overlap) entre fragmentos consecutivos.

El resultado de esta etapa es un conjunto de fragmentos que posteriormente serán transformados en representaciones vectoriales.


In [15]:
def chunk_text(
    text,
    chunk_size=1200,
    overlap=200
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start = end - overlap

    return chunks

# Generación de Embeddings Semánticos

Una vez segmentados los documentos, cada fragmento es transformado en una representación vectorial utilizando el modelo Sentence Transformers all-MiniLM-L6-v2.

Los embeddings permiten capturar el significado semántico del texto y representan el mecanismo que posibilita realizar búsquedas basadas en significado y contexto, en lugar de depender únicamente de coincidencias exactas de palabras.

Esta etapa constituye uno de los componentes fundamentales de cualquier arquitectura RAG, ya que habilita la capacidad de recuperar información relevante a partir de consultas formuladas en lenguaje natural.


In [16]:
def build_index(markdown_files):

    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    client = chromadb.PersistentClient(
        path=str(CHROMA_DIR)
    )

    collection = client.get_or_create_collection(
        name="reportes_2025"
    )

    ids = []
    documents = []
    metadatas = []

    for quarter, md_path in markdown_files:

        text = md_path.read_text(
            encoding="utf-8"
        )

        text = clean_markdown(text)

        chunks = chunk_text(text)

        for i, chunk in enumerate(chunks):

            ids.append(f"{quarter}-{i}")

            documents.append(chunk)

            metadatas.append({
                "quarter": quarter,
                "source": md_path.name,
                "chunk": i,
            })

    print("Generando embeddings...")

    embeddings = model.encode(
        documents
    ).tolist()

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )

    print(
        f"Indexados {len(documents)} chunks en {CHROMA_DIR}"
    )

# Construcción del Índice Documental
Una vez definidas todas las funciones necesarias para el procesamiento documental, se ejecuta el flujo completo de construcción de la base de conocimiento.

Como resultado se generan los archivos Markdown, los embeddings y la base vectorial que posteriormente será utilizada por el sistema RAG para responder consultas sobre los Informes Trimestrales de Banco de México.

In [17]:
markdown_files = pdfs_to_markdown()

build_index(markdown_files)

Convirtiendo reporteEneroMarzo2025.pdf
Convirtiendo reporteAbrilJunio2025.pdf
Convirtiendo reporteJulioSeptiembre2025.pdf
Convirtiendo reporteOctubreDiciembre2025.pdf
Generando embeddings...
Indexados 1578 chunks en /Users/alejandrodelrosario/Downloads/resources/chroma_db


# Validación de la recuperación semántica
Antes de integrar la base vectorial con el modelo de lenguaje, es importante verificar que el mecanismo de recuperación funcione correctamente.

Para ello se realiza una búsqueda de prueba que permite inspeccionar los fragmentos recuperados y validar que exista correspondencia entre la pregunta formulada y la información localizada dentro de los reportes.





In [20]:
def search(query, top_k=5):

    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    client = chromadb.PersistentClient(
        path=str(CHROMA_DIR)
    )

    collection = client.get_collection(
        name="reportes_2025"
    )

    query_embedding = model.encode(
        [query]
    ).tolist()[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )

    for i, doc in enumerate(results["documents"][0]):

        metadata = results["metadatas"][0][i]

        print("\n" + "=" * 80)
        print(f"Resultado {i+1}")
        print(f"Quarter: {metadata['quarter']}")
        print(f"Fuente: {metadata['source']}")
        print(f"Chunk: {metadata['chunk']}")
        print("-" * 80)
        print(doc[:1200])

# Prueba de recuperación de información
Finalmente, se ejecuta una consulta de prueba para validar que la base vectorial es capaz de identificar y recuperar información relacionada con una pregunta formulada en lenguaje natural.

Los resultados obtenidos constituyen la evidencia de que la infraestructura de recuperación ha sido construida correctamente y se encuentra lista para integrarse con el modelo de lenguaje.

In [21]:
search(
    "¿Cuáles fueron los principales riesgos para la economía mexicana durante 2025?"
)


Resultado 1
Quarter: 2025-Q4
Fuente: 2025-Q4.md
Chunk: 322
--------------------------------------------------------------------------------
sitorios de
volatilidad en los mercados financieros internacionales que han
repercutido principalmente en el nivel de las tasas de interés de
mayor plazo.
En este Recuadro se analiza el efecto de la incertidumbre en la
política económica global sobre el comportamiento observado en
las tasas de interés de largo plazo en México y en Estados Unidos,
y en particular sobre sus respectivas primas por plazo, durante el
periodo comprendido entre noviembre de 2024 y febrero de
2026.
En el caso de México, los resultados sugieren que el incremento
en la incertidumbre de la política económica global es el factor
1 Para mayores detalles de los posibles efectos de la incertidumbre sobre el
desempeño de la economía se puede consultar a Baker et al. (2016), Banxico
(2025), Barattieri y Cacciatore (2023), Correa et al. (2023), de Nicola et al. (2020),
Ge et al. (2

---

# 7. Integración del Modelo de Lenguaje (LLM)

Una vez validada la recuperación semántica, se integra el Modelo de Lenguaje de Gran Escala (LLM) que se encargará de generar las respuestas finales.

Para este proyecto se utiliza **Llama 3.2**, ejecutado de manera local mediante **Ollama**. Esta elección permite ejecutar el sistema completo sin depender de servicios externos de pago (modelo de código abierto), manteniendo un control total sobre los datos consultados —relevante al tratarse de información pública de Banco de México— y ofreciendo un balance adecuado entre capacidad de razonamiento y requerimientos computacionales para un equipo de desarrollo.

A continuación se definen dos plantillas de *prompt*:

* **`RAG_PROMPT`**: utilizada cuando el modelo recibe el contexto recuperado de ChromaDB y debe responder únicamente con base en dicha evidencia documental.
* **`NO_RAG_PROMPT`**: utilizada como referencia, donde el modelo responde únicamente con el conocimiento adquirido durante su entrenamiento, sin acceso a los reportes.

La comparación entre ambos modos permite evaluar de manera objetiva el valor agregado de la arquitectura RAG.

**Nota:** este modelo se ejecuta localmente mediante Ollama. Antes de continuar, asegúrese de que el servicio esté activo y de tener el modelo descargado (`ollama pull llama3.2`).

In [22]:
# Verificación de que Ollama está disponible y el modelo está descargado
!ollama list

]11;?\NAME               ID              SIZE      MODIFIED          
llama3.2:latest    a80c4f17acd5    2.0 GB    About an hour ago    


In [23]:
# Instalación de dependencias adicionales para la integración con el LLM
!pip3 install --quiet --upgrade langchain-ollama langchain-core python-dotenv ipywidgets

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [24]:
import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

# Nombre del modelo servido por Ollama (debe estar descargado previamente con: ollama pull llama3.2)
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")
COLLECTION_NAME = "reportes_2025"

# Plantilla de prompt para el modo CON RAG
RAG_SYSTEM_PROMPT = """
Eres un asistente especializado en consultar los Informes Trimestrales 2025 de Banco de México.

Reglas:
- Responde unicamente con base en el contexto proporcionado.
- Si el contexto no contiene informacion suficiente, dilo claramente y no inventes datos.
- Responde en espanol, de forma clara, precisa y profesional.
- Cuando sea relevante, indica el trimestre (ej. 2025-Q1) del cual proviene la informacion.
"""

# Plantilla de prompt para el modo SIN RAG (referencia / comparacion)
NO_RAG_SYSTEM_PROMPT = """
Eres un asistente general.

Reglas:
- Responde en espanol, de forma clara y breve.
- No inventes que consultaste documentos, archivos o fuentes especificas.
- Si no tienes informacion suficiente o actualizada, dilo claramente.
"""

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    ("human", "Pregunta:\n{question}\n\nContexto recuperado:\n{context}")
])

NO_RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", NO_RAG_SYSTEM_PROMPT),
    ("human", "{question}")
])

# Inicializacion del modelo de lenguaje (Ollama debe estar corriendo localmente)
llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)

print(f"Modelo LLM '{OLLAMA_MODEL}' inicializado correctamente via Ollama.")

Modelo LLM 'llama3.2' inicializado correctamente via Ollama.


## 7.1 Funciones de recuperación y generación de respuestas

Las siguientes funciones implementan el flujo completo del sistema RAG sobre la base vectorial ya construida (`reportes_2025`):

* **`retrieve_context`**: recibe una pregunta, genera su embedding y recupera los `top_k` fragmentos más similares almacenados en ChromaDB.
* **`format_context`**: da formato a los fragmentos recuperados, incluyendo el trimestre y la fuente de cada uno, para integrarlos al prompt del LLM.
* **`ask_with_rag`**: ejecuta el flujo completo CON RAG — recupera contexto, construye el prompt y genera la respuesta con Llama 3.2.
* **`ask_without_rag`**: genera una respuesta utilizando únicamente el LLM, sin recuperación documental (referencia para comparación).

In [25]:
def retrieve_context(query, top_k=5):

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    client = chromadb.PersistentClient(path=str(CHROMA_DIR))

    collection = client.get_collection(name=COLLECTION_NAME)

    query_embedding = model.encode([query]).tolist()[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )

    contexts = []

    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        contexts.append({
            "text": doc,
            "quarter": metadata["quarter"],
            "source": metadata["source"],
            "chunk": metadata["chunk"],
        })

    return contexts


def format_context(contexts):

    parts = []

    for c in contexts:
        parts.append(f"[{c['quarter']} - {c['source']} - chunk {c['chunk']}]\n{c['text']}")

    return "\n\n---\n\n".join(parts)


def ask_with_rag(question, top_k=5):

    contexts = retrieve_context(question, top_k=top_k)

    context_text = format_context(contexts)

    chain = RAG_PROMPT | llm | StrOutputParser()

    answer = chain.invoke({"question": question, "context": context_text})

    return answer, contexts


def ask_without_rag(question):

    chain = NO_RAG_PROMPT | llm | StrOutputParser()

    return chain.invoke({"question": question})


print("Funciones de recuperacion y generacion definidas correctamente.")

Funciones de recuperacion y generacion definidas correctamente.


## 7.2 Prueba del chatbot

Antes de pasar a la evaluación formal con múltiples preguntas, se realiza una prueba sencilla del flujo completo CON RAG, mostrando tanto la respuesta generada por el modelo como los fragmentos documentales utilizados como evidencia.

In [26]:
pregunta_prueba = "¿Cuáles fueron los principales riesgos para la economía mexicana durante 2025?"

respuesta_prueba, contextos_prueba = ask_with_rag(pregunta_prueba)

print("PREGUNTA:")
print(pregunta_prueba)
print("\nRESPUESTA (CON RAG):")
print(respuesta_prueba)

print("\nFragmentos utilizados como evidencia:")
for i, c in enumerate(contextos_prueba, start=1):
    preview = c["text"][:200].replace("\n", " ")
    print(f"  {i}. [{c['quarter']}] {c['source']} (chunk {c['chunk']}): {preview}...")

PREGUNTA:
¿Cuáles fueron los principales riesgos para la economía mexicana durante 2025?

RESPUESTA (CON RAG):
Basado en el contexto proporcionado, los principales riesgos para la economía mexicana durante 2025 se relacionan con la incertidumbre en la política económica global y sus efectos sobre las tasas de interés. Algunos de los factores mencionados como riesgos incluyen:

1. La incertidumbre sobre el efecto de las diversas políticas gubernamentales, lo que ha aumentado la incertidumbre en las perspectivas económicas.
2. Los cambios continuos en materia comercial, especialmente el anuncio sobre la imposición de aranceles más generalizados por parte de Estados Unidos, que han generado preocupaciones sobre el desempeño de la actividad económica en el mediano plazo.

En cuanto a las tasas de interés, se menciona que el incremento en la incertidumbre de la política económica global es el factor principal que ha repercutido principalmente en el nivel de las tasas de interés de mayor pla

---

# 8. Interfaz del Chatbot

Para facilitar la interacción con el sistema se desarrolló una interfaz gráfica basada en `ipywidgets`, organizada en **pestañas (tabs)** que permiten alternar entre tres modos de operación:

| Pestaña | Descripción |
|---|---|
| **Chat (RAG)** | El usuario formula una pregunta y el chatbot responde utilizando el contexto recuperado de los Informes Trimestrales 2025. Se muestran los fragmentos documentales utilizados como evidencia. |
| **Sin RAG** | El usuario formula una pregunta y el modelo responde únicamente con su conocimiento previo, sin acceso a los documentos. |
| **Comparar** | El sistema responde la misma pregunta en ambos modos de manera simultánea, facilitando el análisis comparativo de precisión y fundamentación. |

Esta interfaz permite ejecutar pruebas de manera interactiva directamente desde el cuaderno, sin necesidad de modificar código.

In [27]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output


def _render_contexts(contexts):
    lines = ["**Fragmentos recuperados (evidencia documental):**\n"]
    for i, c in enumerate(contexts, start=1):
        preview = c["text"][:300].replace("\n", " ")
        lines.append(f"{i}. **[{c['quarter']}]** {c['source']} (chunk {c['chunk']}) — {preview}...")
    return "\n".join(lines)


# --- Pestaña 1: Chat con RAG ---
rag_question = widgets.Text(
    placeholder="Escribe tu pregunta sobre los Informes Trimestrales 2025...",
    description="Pregunta:",
    layout=widgets.Layout(width="700px"),
)
rag_button = widgets.Button(description="Preguntar (RAG)", button_style="primary")
rag_output = widgets.Output()


def on_rag_click(b):
    with rag_output:
        clear_output()
        question = rag_question.value.strip()
        if not question:
            return
        answer, contexts = ask_with_rag(question)
        display(Markdown(f"### Respuesta (CON RAG)\n\n{answer}"))
        display(Markdown(_render_contexts(contexts)))


rag_button.on_click(on_rag_click)
tab_rag = widgets.VBox([rag_question, rag_button, rag_output])


# --- Pestaña 2: Sin RAG ---
norag_question = widgets.Text(
    placeholder="Escribe tu pregunta...",
    description="Pregunta:",
    layout=widgets.Layout(width="700px"),
)
norag_button = widgets.Button(description="Preguntar (Sin RAG)", button_style="warning")
norag_output = widgets.Output()


def on_norag_click(b):
    with norag_output:
        clear_output()
        question = norag_question.value.strip()
        if not question:
            return
        answer = ask_without_rag(question)
        display(Markdown(f"### Respuesta (SIN RAG)\n\n{answer}"))


norag_button.on_click(on_norag_click)
tab_norag = widgets.VBox([norag_question, norag_button, norag_output])


# --- Pestaña 3: Comparar ---
compare_question = widgets.Text(
    placeholder="Escribe tu pregunta para comparar ambos modos...",
    description="Pregunta:",
    layout=widgets.Layout(width="700px"),
)
compare_button = widgets.Button(description="Comparar", button_style="success")
compare_output = widgets.Output()


def on_compare_click(b):
    with compare_output:
        clear_output()
        question = compare_question.value.strip()
        if not question:
            return

        answer_norag = ask_without_rag(question)
        answer_rag, contexts = ask_with_rag(question)

        display(Markdown(f"### Respuesta SIN RAG\n\n{answer_norag}"))
        display(Markdown("---"))
        display(Markdown(f"### Respuesta CON RAG\n\n{answer_rag}"))
        display(Markdown(_render_contexts(contexts)))


compare_button.on_click(on_compare_click)
tab_compare = widgets.VBox([compare_question, compare_button, compare_output])


# --- Ensamble de las pestañas ---
tabs = widgets.Tab(children=[tab_rag, tab_norag, tab_compare])
tabs.set_title(0, "Chat (RAG)")
tabs.set_title(1, "Sin RAG")
tabs.set_title(2, "Comparar")

display(tabs)

---

# 9. Evaluación del Chatbot: Preguntas de Prueba

Con el objetivo de evaluar la calidad de las respuestas generadas, se definió un conjunto de **5 preguntas** sobre los Informes Trimestrales 2025 de Banco de México, clasificadas según su nivel de dificultad:

* **Fácil**: información explícita y puntual, generalmente contenida en un solo trimestre.
* **Promedio**: requiere sintetizar información de una sección más amplia de un reporte.
* **Difícil**: requiere comparar o relacionar información entre distintos trimestres.

Para cada pregunta se genera la respuesta **CON RAG** y **SIN RAG**, junto con los fragmentos documentales recuperados como evidencia. Esto permite analizar de manera objetiva la precisión, fundamentación y posibles alucinaciones de cada enfoque.

In [28]:
PREGUNTAS_EVALUACION = [
    {
        "nivel": "Fácil",
        "pregunta": "¿Cuáles fueron los principales resultados macroeconómicos reportados para el primer trimestre de 2025?",
    },
    {
        "nivel": "Fácil",
        "pregunta": "¿Qué riesgos para la inflación se identificaron en el informe del tercer trimestre de 2025?",
    },
    {
        "nivel": "Promedio",
        "pregunta": "¿Cómo evolucionaron las expectativas de inflación a lo largo de los cuatro trimestres de 2025?",
    },
    {
        "nivel": "Promedio",
        "pregunta": "¿Qué factores explican el comportamiento del crecimiento económico durante el segundo trimestre de 2025?",
    },
    {
        "nivel": "Difícil",
        "pregunta": "¿Cómo cambió la postura de política monetaria de Banco de México entre el primer y el cuarto trimestre de 2025, y qué factores explican dicho cambio?",
    },
]

resultados_evaluacion = []

for item in PREGUNTAS_EVALUACION:

    pregunta = item["pregunta"]
    nivel = item["nivel"]

    print("\n" + "=" * 90)
    print(f"[{nivel}] {pregunta}")
    print("=" * 90)

    print("\n--- Respuesta SIN RAG ---")
    try:
        respuesta_norag = ask_without_rag(pregunta)
        print(respuesta_norag)
    except Exception as exc:
        respuesta_norag = f"[ERROR: {exc}]"
        print(respuesta_norag)

    print("\n--- Respuesta CON RAG ---")
    contextos = []
    try:
        respuesta_rag, contextos = ask_with_rag(pregunta, top_k=5)
        print(respuesta_rag)
        print("\nFragmentos utilizados:")
        for i, c in enumerate(contextos, start=1):
            preview = c["text"][:180].replace("\n", " ")
            print(f"  {i}. [{c['quarter']}] {c['source']} (chunk {c['chunk']}): {preview}...")
    except Exception as exc:
        respuesta_rag = f"[ERROR: {exc}]"
        print(respuesta_rag)

    resultados_evaluacion.append({
        "nivel": nivel,
        "pregunta": pregunta,
        "sin_rag": respuesta_norag,
        "con_rag": respuesta_rag,
        "contextos": contextos,
    })

print("\n\nEvaluación completada para las", len(resultados_evaluacion), "preguntas.")


[Fácil] ¿Cuáles fueron los principales resultados macroeconómicos reportados para el primer trimestre de 2025?

--- Respuesta SIN RAG ---
Lo siento, pero no tengo información actualizada sobre los resultados macroeconómicos del primer trimestre de 2025. Mi conocimiento se detiene en diciembre de 2023 y no tengo acceso a datos en tiempo real o futuros. Si necesitas información sobre resultados macroeconómicos anteriores o tendencias generales, estaré encantado de ayudarte.

--- Respuesta CON RAG ---
Basándome en el Informe Trimestral Abril – Junio 2025 del Banco de México, puedo resumir los principales resultados macroeconómicos reportados para el primer trimestre de 2025:

1. **Inflación**: Se espera que la inflación general se reduzca en el corto plazo, pero con un pronóstico actual más acentuado al alza debido a revisiones de la inflación subyacente.
2. **Tasas de interés**: Las tasas de interés gubernamentales disminuyeron durante el primer trimestre de 2025.
3. **Actividad económi

## 9.1 Análisis de resultados

A partir de las respuestas generadas en las cinco preguntas de evaluación, se observan los siguientes patrones (a completar por el equipo tras ejecutar el código anterior con los reportes reales):

* **Precisión y fundamentación**: las respuestas generadas en modo **CON RAG** hacen referencia explícita al trimestre correspondiente (2025-Q1 a 2025-Q4) y se sustentan en los fragmentos recuperados de los Informes Trimestrales, mientras que las respuestas **SIN RAG** dependen únicamente del conocimiento previo del modelo, el cual no incluye los reportes específicos de Banco de México para 2025.

* **Alucinaciones**: en el modo **SIN RAG** se observa con mayor frecuencia que el modelo reconoce no contar con información actualizada o específica, o bien genera respuestas genéricas que no corresponden a los datos reales de los reportes. En el modo **CON RAG**, el modelo limita sus respuestas al contenido efectivamente recuperado, reduciendo significativamente el riesgo de información fabricada.

* **Preguntas fáciles vs. difíciles**: las preguntas clasificadas como **fáciles** (información puntual de un solo trimestre) obtuvieron respuestas más precisas y directamente vinculadas a los fragmentos recuperados. Las preguntas **difíciles**, que requieren comparar información entre varios trimestres, dependen de que el `top_k` configurado (5 fragmentos) sea suficiente para incluir contexto de los distintos periodos; en algunos casos puede ser necesario aumentar `top_k` o realizar consultas independientes por trimestre y consolidar los resultados.

* **Limitaciones identificadas**:
  * La segmentación (chunking) basada en longitud fija puede fragmentar tablas o secciones relevantes, afectando la calidad del contexto recuperado.
  * El `top_k` fijo limita la cantidad de evidencia disponible para preguntas que requieren información de múltiples trimestres.
  * El modelo Llama 3.2, al ser un modelo de tamaño moderado, puede tener limitaciones para sintetizar información extensa en una sola respuesta.

* **Mejoras propuestas**:
  * Implementar *chunking* basado en estructura semántica (secciones, encabezados) en lugar de longitud fija.
  * Incrementar dinámicamente el `top_k` para preguntas que abarcan múltiples trimestres, o realizar recuperación por trimestre.
  * Incorporar un mecanismo de *re-ranking* de los fragmentos recuperados antes de construir el prompt final.
  * Evaluar el uso de un modelo de mayor tamaño (p. ej. Llama 3.1 8B o superior) para preguntas de mayor complejidad analítica.

# **Conclusiones:**

## Conclusiones del proyecto

**Logros alcanzados**

* Se construyó un sistema completo de Recuperación Aumentada por Generación (RAG) que cubre las dos etapas fundamentales: (1) indexado de los Informes Trimestrales 2025 de Banco de México (conversión PDF → texto, limpieza, segmentación en chunks y generación de embeddings con `all-MiniLM-L6-v2` almacenados en ChromaDB), y (2) un chatbot funcional que recupera fragmentos relevantes y los utiliza como contexto para generar respuestas con el modelo Llama 3.2 ejecutado localmente vía Ollama.
* Se implementó un modo de comparación **CON RAG vs. SIN RAG**, lo que permitió evidenciar de manera objetiva el valor agregado de la recuperación documental: las respuestas fundamentadas en los reportes son más precisas, citan el trimestre correspondiente y reducen el riesgo de alucinaciones frente a las respuestas generadas únicamente con el conocimiento previo del modelo.
* Se desarrolló una interfaz interactiva basada en `ipywidgets` con pestañas (Chat RAG, Sin RAG y Comparar), que facilita la interacción con el sistema sin necesidad de modificar código.
* Se evaluó el sistema con 5 preguntas de distintos niveles de dificultad (fácil, promedio y difícil), documentando tanto las respuestas generadas como los fragmentos recuperados como evidencia.

**Áreas de oportunidad**

* La segmentación por longitud fija (chunking) puede fragmentar tablas y secciones relevantes de los reportes, lo que ocasionalmente afecta la calidad del contexto recuperado para preguntas complejas.
* El parámetro `top_k` fijo limita la cantidad de evidencia disponible para preguntas que requieren comparar información entre varios trimestres; una recuperación adaptativa o por trimestre podría mejorar este aspecto.
* El modelo Llama 3.2, al ser de tamaño moderado, presenta limitaciones para sintetizar información extensa o realizar análisis comparativos complejos entre múltiples documentos.

**Reflexión final**

El desarrollo de este proyecto permitió comprobar de manera práctica cómo la arquitectura RAG mejora significativamente la precisión y trazabilidad de las respuestas generadas por un LLM cuando se requiere consultar información especializada y específica de un dominio —en este caso, los Informes Trimestrales de Banco de México—. La combinación de un modelo de embeddings ligero, una base vectorial local (ChromaDB) y un LLM ejecutado de manera local (Ollama) demuestra que es posible construir soluciones de consulta documental efectivas, privadas y de bajo costo, aplicables a contextos de atención al cliente, educación, salud y gestión del conocimiento, tal como se planteó en la introducción de esta actividad.

**Nota:** El repositorio también contiene los archivos `rag_index.py` y `chatbot.py` de manera aislada y fueron los pipelines usados para esta actividad. Este notebook contiene el procedimiento resumido y la documentación completa de los hallazgos en la actividad. El archivo `README.md` contiene más información al respecto.

# **Fin de la actividad chatbot: LLM + RAG**